# w2-analysis — Week 2: Dictionary Regressions & Validation

**Project:** Making Taste Legible: Symbolic Boundaries and Expert Valuation in Whisky Reviews

Four tasks in one notebook:
- **Task A:** Dictionary-score OLS regressions (full, without explicit eval, section-level)
- **Task B:** Known-groups validation (Islay/peated, sherry, bourbon, high-score, low-score)
- **Task C:** Generic sentiment comparison (VADER)
- **Task D:** Close-reading candidate selection

## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
from scipy import stats
import json, re, os, warnings
warnings.filterwarnings('ignore')

os.chdir('/Users/mac/Desktop/CSSS594/FinalProject')
os.makedirs('figures', exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
                      'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight'})
PALETTE = sns.color_palette('tab10')

def save_fig(name):
    for ext in ['pdf', 'png']:
        plt.savefig(f'figures/{name}.{ext}', bbox_inches='tight')
    plt.close()

# --------------------------------------------------------------------
# Load and merge
# --------------------------------------------------------------------
df = pd.read_parquet('data/whiskyfun_tokenized.parquet')
feat = pd.read_parquet('data/whiskyfun_dict_features.parquet')
with open('data/whiskyfun_dictionary_v1.json') as f:
    dictionary = json.load(f)
print(f"Loaded: {len(df):,} reviews, {feat.shape[1]} feature columns, {dictionary['total_terms']} dict terms")

# Drop duplicate metadata columns from feat before merge
meta_overlap = ['score', 'review_year', 'distillery', 'review_length',
                'identity_status', 'match_source']
feat_to_merge = feat.drop(columns=[c for c in meta_overlap if c in feat.columns])
df = df.merge(feat_to_merge, on='dedupe_hash', how='left')
print(f"Merged: {len(df)} rows x {len(df.columns)} columns")

# Category info
CAT_KEYS = list(dictionary['categories'].keys())
CAT_SHORT = {
    'fruit_aromatics': 'fruit', 'peat_smoke_coastal': 'peat',
    'sherry_rancio_oxidative': 'sherry', 'oak_cask_wood': 'oak',
    'texture_body': 'texture', 'mineral_earth_farmy': 'mineral',
    'flaws_off_notes': 'flaw', 'finish_complexity_balance': 'structure',
    'explicit_evaluation': 'eval'
}
SHORT_CATS = [CAT_SHORT[c] for c in CAT_KEYS]
SCOPES = ['review_text', 'nose', 'mouth', 'finish', 'comments', 'nmf']
SCOPE_LABELS = ['Full Text', 'Nose', 'Mouth', 'Finish', 'Comments', 'NMF']

Loaded: 11,149 reviews, 241 feature columns, 281 dict terms
Merged: 11149 rows x 251 columns


## Task A: Dictionary-Score Regressions

### A1. Full Dictionary OLS Regression

In [2]:
# Primary model: all 9 categories on review_text scope, with review_length + year FE
formula_full = (
    'score ~ fruit_review_text_per1k + peat_review_text_per1k + '
    'sherry_review_text_per1k + oak_review_text_per1k + '
    'texture_review_text_per1k + mineral_review_text_per1k + '
    'flaw_review_text_per1k + structure_review_text_per1k + '
    'eval_review_text_per1k + review_length + C(review_year)'
)

m_full = smf.ols(formula_full, data=df).fit()
print(m_full.summary().tables[1])
print(f"Adj R²: {m_full.rsquared_adj:.4f}, R²: {m_full.rsquared:.4f}, N: {int(m_full.nobs)}")

                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      78.7143      0.293    268.207      0.000      78.139      79.290
C(review_year)[T.2013]          0.1643      0.249      0.659      0.510      -0.324       0.653
C(review_year)[T.2014]          0.6436      0.247      2.606      0.009       0.160       1.128
C(review_year)[T.2015]         -0.1375      0.253     -0.542      0.588      -0.634       0.359
C(review_year)[T.2016]          0.7659      0.261      2.937      0.003       0.255       1.277
C(review_year)[T.2017]          0.5998      0.241      2.487      0.013       0.127       1.073
C(review_year)[T.2018]          0.4459      0.238      1.877      0.061      -0.020       0.911
C(review_year)[T.2019]          0.8248      0.247      3.344      0.001       0.341       1.308
C(review_year)[T.2020]          0.8094  

### A2. Without Explicit Evaluation

In [3]:
# Drop eval category — tests whether sensory/structural vocabulary retains signal
formula_no_eval = (
    'score ~ fruit_review_text_per1k + peat_review_text_per1k + '
    'sherry_review_text_per1k + oak_review_text_per1k + '
    'texture_review_text_per1k + mineral_review_text_per1k + '
    'flaw_review_text_per1k + structure_review_text_per1k + '
    'review_length + C(review_year)'
)

m_no_eval = smf.ols(formula_no_eval, data=df).fit()
print(m_no_eval.summary().tables[1])
print(); print(f"Adj R²: {m_no_eval.rsquared_adj:.4f}, R²: {m_no_eval.rsquared:.4f}")
delta_r2 = m_full.rsquared_adj - m_no_eval.rsquared_adj
print(f"ΔAdj-R² (full - no_eval): {delta_r2:+.4f}  ← eval category contribution")

                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      80.4429      0.285    282.602      0.000      79.885      81.001
C(review_year)[T.2013]          0.1259      0.253      0.497      0.619      -0.371       0.623
C(review_year)[T.2014]          0.6088      0.251      2.424      0.015       0.116       1.101
C(review_year)[T.2015]         -0.1672      0.258     -0.649      0.517      -0.673       0.338
C(review_year)[T.2016]          0.7851      0.265      2.960      0.003       0.265       1.305
C(review_year)[T.2017]          0.5483      0.245      2.235      0.025       0.067       1.029
C(review_year)[T.2018]          0.4957      0.242      2.052      0.040       0.022       0.969
C(review_year)[T.2019]          0.8311      0.251      3.313      0.001       0.339       1.323
C(review_year)[T.2020]          0.7401  

### A3. Section-Level Comparisons

In [4]:
# Run full-dictionary model on each scope, using wordcount_{scope} as length control
# Drop rows where wordcount_{scope} == 0
section_results = []
scope_formulas = {}

for scope, label in zip(SCOPES, SCOPE_LABELS):
    wc_col = f'wordcount_{scope}'
    # Filter to rows with text in this scope
    scope_df = df[df[wc_col] > 0].copy()

    formula = (
        f'score ~ fruit_{scope}_per1k + peat_{scope}_per1k + '
        f'sherry_{scope}_per1k + oak_{scope}_per1k + '
        f'texture_{scope}_per1k + mineral_{scope}_per1k + '
        f'flaw_{scope}_per1k + structure_{scope}_per1k + '
        f'eval_{scope}_per1k + {wc_col} + C(review_year)'
    )
    scope_formulas[scope] = formula

    try:
        m = smf.ols(formula, data=scope_df).fit()
        section_results.append({
            'Scope': label,
            'N': int(m.nobs),
            'Adj_R2': round(m.rsquared_adj, 4),
            'R2': round(m.rsquared, 4)
        })
        print(f"{label:<12s}: Adj-R²={m.rsquared_adj:.4f}, N={int(m.nobs)}")
    except Exception as e:
        print(f"{label}: FAILED — {e}")

# Baseline: length + year FE only (review_text scope)
formula_baseline = 'score ~ review_length + C(review_year)'
m_baseline = smf.ols(formula_baseline, data=df).fit()
section_results.append({
    'Scope': 'Baseline (length+year)', 'N': int(m_baseline.nobs),
    'Adj_R2': round(m_baseline.rsquared_adj, 4), 'R2': round(m_baseline.rsquared, 4)
})
print(); print(f"Baseline (length+year): Adj-R²={m_baseline.rsquared_adj:.4f}")

sr_df = pd.DataFrame(section_results)
print(); print("=== R² Comparison Across Scopes ===")
print(sr_df.to_string(index=False))

# Figure: R² comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(sr_df)), sr_df['Adj_R2'], color=[PALETTE[i % 10] for i in range(len(sr_df))], edgecolor='white')
ax.set_xticks(range(len(sr_df))); ax.set_xticklabels(sr_df['Scope'], rotation=30, ha='right')
ax.set_ylabel('Adjusted R²'); ax.set_title('Figure W2-1: R² by Text Scope (Full Dictionary Model)')
for bar, val in zip(bars, sr_df['Adj_R2']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{val:.3f}', ha='center', fontsize=9)
plt.tight_layout(); save_fig('fig_w2_a3_r2_by_scope')

Full Text   : Adj-R²=0.2728, N=11149
Nose        : Adj-R²=0.1977, N=11133
Mouth       : Adj-R²=0.2276, N=11130
Finish      : Adj-R²=0.1344, N=11120
Comments    : Adj-R²=0.0274, N=11100
NMF         : Adj-R²=0.3207, N=11133

Baseline (length+year): Adj-R²=0.1031

=== R² Comparison Across Scopes ===
                 Scope     N  Adj_R2     R2
             Full Text 11149  0.2728 0.2743
                  Nose 11133  0.1977 0.1994
                 Mouth 11130  0.2276 0.2292
                Finish 11120  0.1344 0.1362
              Comments 11100  0.0274 0.0295
                   NMF 11133  0.3207 0.3221
Baseline (length+year) 11149  0.1031 0.1042


### A4. Regression Coefficient Table

In [5]:
# Build clean coefficient table from the full model
coef_table = pd.DataFrame({
    'Coefficient': m_full.params,
    'SE': m_full.bse,
    't': m_full.tvalues,
    'p': m_full.pvalues
})

# Add significance stars
def stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return ''

coef_table['sig'] = [stars(p) for p in coef_table['p']]
coef_table['Coef (SE)'] = [
    f"{c:.3f}{s} ({se:.3f})" for c, se, s in zip(coef_table['Coefficient'], coef_table['SE'], coef_table['sig'])
]

# Keep only category features + review_length
cat_rows = [r for r in coef_table.index if 'per1k' in r or 'review_length' in r]
print("=== Full Dictionary Model: Category Coefficients ===")
print(coef_table.loc[cat_rows, ['Coef (SE)', 't', 'p']].to_string())

# Save
coef_table.to_csv('data/w2_regression_results.csv')
print(); print("Saved: data/w2_regression_results.csv")
sr_df.to_csv('data/w2_r2_by_scope.csv', index=False)

=== Full Dictionary Model: Category Coefficients ===
                                     Coef (SE)          t              p
fruit_review_text_per1k       0.032*** (0.003)   9.579000   1.187603e-21
peat_review_text_per1k        0.060*** (0.003)  20.067172   5.107152e-88
sherry_review_text_per1k      0.039*** (0.003)  11.173393   7.835336e-29
oak_review_text_per1k        -0.056*** (0.004) -12.374152   6.121009e-35
texture_review_text_per1k     0.030*** (0.005)   6.621518   3.719043e-11
mineral_review_text_per1k    -0.026*** (0.004)  -7.082309   1.503274e-12
flaw_review_text_per1k       -0.221*** (0.007) -31.044757  5.241916e-203
structure_review_text_per1k   0.029*** (0.009)   3.388244   7.058555e-04
eval_review_text_per1k        0.095*** (0.005)  19.575580   6.396438e-84
review_length                 0.031*** (0.001)  32.897365  1.378166e-226

Saved: data/w2_regression_results.csv


## Task B: Known-Groups Validation

### B1. Islay / Peated Groups

In [6]:
# Method A: Distillery-based
ISLAY_DISTS = ['Ardbeg', 'Bowmore', 'Bruichladdich', 'Bunnahabhain',
               'Caol Ila', 'Kilchoman', 'Laphroaig', 'Port Ellen']
df['islay_dist'] = df['distillery'].isin(ISLAY_DISTS)

# Method B: Feature-based (preferred — captures peated style regardless of distillery)
peat_75 = df['peat_review_text_per1k'].quantile(0.75)
df['is_peated'] = df['peat_review_text_per1k'] > peat_75

print(f"Islay by distillery: {df['islay_dist'].sum():,} reviews")
print(f"Peated (peat > P75={peat_75:.1f}/1k): {df['is_peated'].sum():,} reviews")
print(f"Overlap: {((df['islay_dist']) & (df['is_peated'])).sum():,} reviews are Islay AND peated")

# Use feature-based for comparisons
def compare_groups(mask, label, cats_to_show=None):
    """Compare dictionary rates between group (mask=True) and rest."""
    if cats_to_show is None:
        cats_to_show = SHORT_CATS

    results = []
    for short in cats_to_show:
        col = f'{short}_review_text_per1k'
        g1 = df.loc[mask, col].dropna()
        g2 = df.loc[~mask, col].dropna()

        # t-test (Welch's) and Mann-Whitney
        t_stat, t_p = stats.ttest_ind(g1, g2, equal_var=False)
        mw_u, mw_p = stats.mannwhitneyu(g1, g2, alternative='two-sided')

        # Cohen's d
        pooled_std = np.sqrt((g1.std()**2 + g2.std()**2) / 2)
        d = (g1.mean() - g2.mean()) / pooled_std if pooled_std > 0 else 0

        results.append({
            'Category': short,
            f'{label}_mean': round(g1.mean(), 2),
            'Other_mean': round(g2.mean(), 2),
            'Diff': round(g1.mean() - g2.mean(), 2),
            "Cohen_d": round(d, 3),
            't_p': round(t_p, 4),
            'sig': stars(t_p)
        })
    return pd.DataFrame(results)

print(); print("=== Peated vs Non-Peated ===")
peat_results = compare_groups(df['is_peated'], 'Peated')
print(peat_results.to_string(index=False))
peat_results.to_csv('data/w2_known_groups_peat.csv', index=False)

Islay by distillery: 2,214 reviews
Peated (peat > P75=15.5/1k): 2,784 reviews
Overlap: 1,343 reviews are Islay AND peated

=== Peated vs Non-Peated ===
 Category  Peated_mean  Other_mean  Diff  Cohen_d    t_p sig
    fruit        13.94       19.02 -5.08   -0.400 0.0000 ***
     peat        31.83        3.13 28.70    2.571 0.0000 ***
   sherry         7.16       10.11 -2.95   -0.241 0.0000 ***
      oak         5.17        8.01 -2.83   -0.311 0.0000 ***
  texture        10.91        9.23  1.68    0.171 0.0000 ***
  mineral        11.01       12.87 -1.86   -0.154 0.0000 ***
     flaw         2.89        3.32 -0.43   -0.071 0.0009 ***
structure         3.52        3.10  0.42    0.082 0.0002 ***
     eval        10.26       10.37 -0.11   -0.012 0.5969    


### B2-B3. Sherry & Bourbon Cask Groups

In [7]:
# Sherry: top quartile of sherry per1k
sherry_75 = df['sherry_review_text_per1k'].quantile(0.75)
df['is_sherried'] = df['sherry_review_text_per1k'] > sherry_75
print(f"Sherried (sherry > P75={sherry_75:.1f}/1k): {df['is_sherried'].sum():,} reviews")

# Bourbon: high oak, low sherry
oak_75 = df['oak_review_text_per1k'].quantile(0.75)
sherry_median = df['sherry_review_text_per1k'].median()
df['is_bourbon'] = (df['oak_review_text_per1k'] > oak_75) & (df['sherry_review_text_per1k'] < sherry_median)
print(f"Bourbon cask (oak>P75={oak_75:.1f}, sherry<median={sherry_median:.1f}): {df['is_bourbon'].sum():,} reviews")

print(); print("=== Sherried vs Non-Sherried ===")
sherry_results = compare_groups(df['is_sherried'], 'Sherried')
print(sherry_results.to_string(index=False))
sherry_results.to_csv('data/w2_known_groups_sherry.csv', index=False)

print(); print("=== Bourbon Cask vs Rest ===")
bourbon_results = compare_groups(df['is_bourbon'], 'Bourbon')
print(bourbon_results.to_string(index=False))
bourbon_results.to_csv('data/w2_known_groups_bourbon.csv', index=False)

Sherried (sherry > P75=13.9/1k): 2,777 reviews
Bourbon cask (oak>P75=10.8, sherry<median=5.3): 1,276 reviews

=== Sherried vs Non-Sherried ===
 Category  Sherried_mean  Other_mean  Diff  Cohen_d    t_p sig
    fruit          16.62       18.13 -1.51   -0.113 0.0000 ***
     peat           7.59       11.20 -3.61   -0.259 0.0000 ***
   sherry          27.90        3.23 24.67    2.613 0.0000 ***
      oak           7.63        7.19  0.45    0.046 0.0364   *
  texture           9.91        9.57  0.34    0.036 0.1027    
  mineral           9.06       13.51 -4.45   -0.382 0.0000 ***
     flaw           3.45        3.14  0.31    0.050 0.0232   *
structure           3.39        3.15  0.25    0.049 0.0261   *
     eval          10.15       10.40 -0.26   -0.028 0.1938    

=== Bourbon Cask vs Rest ===
 Category  Bourbon_mean  Other_mean  Diff  Cohen_d    t_p sig
    fruit         20.86       17.35  3.51    0.245 0.0000 ***
     peat          8.63       10.51 -1.89   -0.125 0.0000 ***
   sherry  

### B4-B5. High-Score & Low-Score Groups

In [8]:
df['high_score'] = df['score'] >= 90
df['low_score'] = df['score'] <= 75
print(f"High-score (>=90): {df['high_score'].sum():,} reviews")
print(f"Low-score (<=75): {df['low_score'].sum():,} reviews")

print(); print("=== High-Score (>=90) vs Rest ===")
high_results = compare_groups(df['high_score'], 'HighScore')
print(high_results.to_string(index=False))
high_results.to_csv('data/w2_known_groups_highscore.csv', index=False)

print(); print("=== Low-Score (<=75) vs Rest ===")
low_results = compare_groups(df['low_score'], 'LowScore')
print(low_results.to_string(index=False))
low_results.to_csv('data/w2_known_groups_lowscore.csv', index=False)

High-score (>=90): 2,951 reviews
Low-score (<=75): 359 reviews

=== High-Score (>=90) vs Rest ===
 Category  HighScore_mean  Other_mean  Diff  Cohen_d    t_p sig
    fruit           16.47       18.22 -1.75   -0.132 0.0000 ***
     peat           15.10        8.57  6.53    0.420 0.0000 ***
   sherry            9.66        9.27  0.39    0.030 0.1542    
      oak            5.14        8.08 -2.94   -0.331 0.0000 ***
  texture           10.38        9.39  0.99    0.101 0.0000 ***
  mineral            9.57       13.42 -3.85   -0.325 0.0000 ***
     flaw            1.73        3.75 -2.01   -0.363 0.0000 ***
structure            3.89        2.96  0.92    0.179 0.0000 ***
     eval           11.36        9.97  1.38    0.150 0.0000 ***

=== Low-Score (<=75) vs Rest ===
 Category  LowScore_mean  Other_mean  Diff  Cohen_d    t_p sig
    fruit          12.44       17.93 -5.49   -0.447 0.0000 ***
     peat           4.44       10.49 -6.06   -0.502 0.0000 ***
   sherry           4.99        9.52 -4

### B6. Known-Groups Summary Figure

In [9]:
# Combined figure: key category differences across all groups
groups_data = {
    'Peated': ('is_peated', ['peat', 'sherry', 'oak', 'flaw']),
    'Sherried': ('is_sherried', ['sherry', 'oak', 'fruit', 'flaw']),
    'Bourbon': ('is_bourbon', ['oak', 'sherry', 'vanilla_not_in_features_use_oak', 'flaw']),
    'High Score': ('high_score', ['structure', 'fruit', 'eval', 'flaw']),
    'Low Score': ('low_score', ['flaw', 'structure', 'fruit', 'eval']),
}

# Simplified version for figure — 4 groups × key categories
groups_for_fig = {
    'Peated': ('is_peated', ['peat', 'flaw', 'structure', 'eval']),
    'Sherried': ('is_sherried', ['sherry', 'peat', 'structure', 'eval']),
    'High Score': ('high_score', ['structure', 'eval', 'flaw', 'peat']),
    'Low Score': ('low_score', ['flaw', 'eval', 'structure', 'peat']),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (gname, (mask_col, cats)) in zip(axes.flat, groups_for_fig.items()):
    x = np.arange(len(cats)); width = 0.35
    mask = df[mask_col]
    g_means = [df.loc[mask, f'{c}_review_text_per1k'].mean() for c in cats]
    o_means = [df.loc[~mask, f'{c}_review_text_per1k'].mean() for c in cats]

    ax.bar(x - width/2, g_means, width, label=gname, color=PALETTE[1], edgecolor='white')
    ax.bar(x + width/2, o_means, width, label='Other', color=PALETTE[5], edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=10)
    ax.set_ylabel('Mean per 1k tokens'); ax.legend(fontsize=9)
    ax.set_title(f'{gname} (n={mask.sum():,})')

plt.suptitle('Figure W2-2: Known-Groups Validation — Dictionary Feature Comparison', fontsize=14)
plt.tight_layout(); save_fig('fig_w2_b_known_groups')

# Combined results table
all_group_results = pd.concat([
    peat_results.assign(Group='Peated'),
    sherry_results.assign(Group='Sherried'),
    bourbon_results.assign(Group='Bourbon'),
    high_results.assign(Group='HighScore'),
    low_results.assign(Group='LowScore')
], ignore_index=True)
all_group_results.to_csv('data/w2_known_groups.csv', index=False)
print("Combined results saved: data/w2_known_groups.csv")

Combined results saved: data/w2_known_groups.csv


## Task C: Generic Sentiment Comparison

### C1. Compute VADER Sentiment

In [10]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Ensure VADER lexicon is available
import nltk
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon', quiet=True)

analyzer = SentimentIntensityAnalyzer()

# Apply to untokenized review text
def vader_scores(text):
    if pd.isna(text) or str(text).strip() == '':
        return pd.Series({'vader_compound': np.nan, 'vader_pos': np.nan,
                          'vader_neg': np.nan, 'vader_neu': np.nan})
    scores = analyzer.polarity_scores(str(text))
    return pd.Series({'vader_compound': scores['compound'],
                      'vader_pos': scores['pos'],
                      'vader_neg': scores['neg'],
                      'vader_neu': scores['neu']})

print("Computing VADER sentiment on review_text_original...")
vader_df = df['review_text_original'].apply(vader_scores)

for col in ['vader_compound', 'vader_pos', 'vader_neg', 'vader_neu']:
    df[col] = vader_df[col]

print(f"VADER compound: mean={df['vader_compound'].mean():.3f}, std={df['vader_compound'].std():.3f}")
print(f"  Positive: mean={df['vader_pos'].mean():.3f}")
print(f"  Negative: mean={df['vader_neg'].mean():.3f}")
print(f"  Neutral:  mean={df['vader_neu'].mean():.3f}")

# Correlation with score
for col in ['vader_compound', 'vader_pos', 'vader_neg']:
    r = df[col].corr(df['score'])
    print(f"  score ~ {col}: r={r:+.4f}")

Computing VADER sentiment on review_text_original...
VADER compound: mean=0.855, std=0.321
  Positive: mean=0.152
  Negative: mean=0.036
  Neutral:  mean=0.812
  score ~ vader_compound: r=+0.2821
  score ~ vader_pos: r=+0.1268
  score ~ vader_neg: r=-0.2469


### C2. Sentiment Regression & R² Comparison

In [11]:
# M0: Baseline (length + year FE)
m0 = smf.ols('score ~ review_length + C(review_year)', data=df).fit()

# M1: VADER sentiment
m1 = smf.ols('score ~ vader_compound + review_length + C(review_year)', data=df).fit()

# Collect R² values
r2_comparison = pd.DataFrame([
    {'Model': 'M0: Baseline (length + year)', 'Adj_R2': round(m0.rsquared_adj, 4), 'R2': round(m0.rsquared, 4), 'N': int(m0.nobs)},
    {'Model': 'M1: VADER Sentiment', 'Adj_R2': round(m1.rsquared_adj, 4), 'R2': round(m1.rsquared, 4), 'N': int(m1.nobs)},
    {'Model': 'M2: Full Dictionary (9 cats)', 'Adj_R2': round(m_full.rsquared_adj, 4), 'R2': round(m_full.rsquared, 4), 'N': int(m_full.nobs)},
    {'Model': 'M3: Dictionary minus Eval (8 cats)', 'Adj_R2': round(m_no_eval.rsquared_adj, 4), 'R2': round(m_no_eval.rsquared, 4), 'N': int(m_no_eval.nobs)},
])

print("=== R² Comparison: Sentiment vs. Dictionary ===")
print(r2_comparison.to_string(index=False))

# Figure
fig, ax = plt.subplots(figsize=(9, 5))
models_short = ['Baseline', 'VADER', 'Full Dict (9)', 'Dict-Eval (8)']
colors = [PALETTE[5], PALETTE[3], PALETTE[0], PALETTE[1]]
bars = ax.bar(models_short, r2_comparison['Adj_R2'], color=colors, edgecolor='white')
ax.set_ylabel('Adjusted R²'); ax.set_title('Figure W2-3: R² Comparison — Generic Sentiment vs. Domain Dictionary')
for bar, val in zip(bars, r2_comparison['Adj_R2']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{val:.4f}', ha='center', fontsize=11)
plt.tight_layout(); save_fig('fig_w2_c_r2_comparison')

# Save updated features with VADER columns to a new parquet
vader_cols = ['dedupe_hash', 'vader_compound', 'vader_pos', 'vader_neg', 'vader_neu']
feat_updated = feat.merge(df[vader_cols], on='dedupe_hash', how='left')
feat_updated.to_parquet('data/whiskyfun_dict_features.parquet', index=False)
print(); print("Updated features saved: data/whiskyfun_dict_features.parquet (now includes VADER columns)")

=== R² Comparison: Sentiment vs. Dictionary ===
                             Model  Adj_R2     R2     N
      M0: Baseline (length + year)  0.1031 0.1042 11149
               M1: VADER Sentiment  0.1608 0.1620 11149
      M2: Full Dictionary (9 cats)  0.2734 0.2749 11149
M3: Dictionary minus Eval (8 cats)  0.2484 0.2499 11149

Updated features saved: data/whiskyfun_dict_features.parquet (now includes VADER columns)


## Task D: Close-Reading Candidate Selection

### D1. Define Selection Functions

In [12]:
def find_reviews(mask, criterion_name, n=3, sort_by=None):
    """Select up to n reviews matching mask, optionally sorted by a column."""
    candidates = df[mask].copy()
    if len(candidates) == 0:
        print(f"  {criterion_name}: NO MATCHES")
        return pd.DataFrame()
    if sort_by:
        candidates = candidates.sort_values(sort_by, ascending=False)
    selected = candidates.head(n).copy()
    selected['selection_criterion'] = criterion_name
    return selected[
        ['dedupe_hash', 'whisky_name_raw', 'distillery', 'score', 'review_text_original',
         'review_year'] +
        [f'{c}_review_text_per1k' for c in SHORT_CATS] +
        ['selection_criterion']
    ]

# Define thresholds
flaw_p90 = df['flaw_review_text_per1k'].quantile(0.90)
fruit_p75 = df['fruit_review_text_per1k'].quantile(0.75)
peat_p75 = df['peat_review_text_per1k'].quantile(0.75)
structure_p75 = df['structure_review_text_per1k'].quantile(0.75)
eval_median = df['eval_review_text_per1k'].median()

print(f"Thresholds: flaw P90={flaw_p90:.1f}, fruit P75={fruit_p75:.1f}, peat P75={peat_p75:.1f}, structure P75={structure_p75:.1f}")

# Term-level regex searches on tokenized review_text
def term_search(terms):
    """Build regex for any of the given terms (word-boundary, case-insensitive)."""
    return df['review_text'].str.contains(
        r'\b(?:' + '|'.join(re.escape(t) for t in terms) + r')\b',
        case=False, regex=True, na=False
    )

has_sulphur = term_search(['sulphur', 'sulfurous', 'sulphuric'])
has_farmyard = term_search(['farmyard', 'farmy', 'barnyard'])
has_overoaked = term_search(['over_oaked', 'planky', 'too_woody', 'over_wooded'])
has_nostalgia = term_search(['old_school', 'old_bottle_effect', 'old_style', 'good_old_days'])

print(f"Reviews mentioning: sulphur={has_sulphur.sum()}, farmyard={has_farmyard.sum()}, overoaked={has_overoaked.sum()}, nostalgia={has_nostalgia.sum()}")

Thresholds: flaw P90=10.6, fruit P75=25.2, peat P75=15.5, structure P75=5.9
Reviews mentioning: sulphur=276, farmyard=319, overoaked=13, nostalgia=465


### D2. Select Candidates by Criterion

In [13]:
selections = []

# D1: High score + high flaw
sel = find_reviews(
    (df['score'] >= 90) & (df['flaw_review_text_per1k'] > flaw_p90),
    'D1: High score + high flaw', n=3, sort_by='flaw_review_text_per1k'
)
selections.append(sel)

# D2: Low score + high fruit
sel = find_reviews(
    (df['score'] <= 75) & (df['fruit_review_text_per1k'] > fruit_p75),
    'D2: Low score + high fruit', n=3, sort_by='fruit_review_text_per1k'
)
selections.append(sel)

# D3: High sulphur + high score
sel = find_reviews(
    has_sulphur & (df['score'] >= 85),
    'D3: Sulphur + high score (context-dependent)', n=3, sort_by='score'
)
selections.append(sel)

# D4a: Smoke valued
sel = find_reviews(
    (df['peat_review_text_per1k'] > peat_p75) & (df['score'] >= 88),
    'D4a: Smoke valued', n=2, sort_by='score'
)
selections.append(sel)

# D4b: Smoke condemned
sel = find_reviews(
    (df['peat_review_text_per1k'] > peat_p75) & (df['score'] <= 78),
    'D4b: Smoke condemned', n=2, sort_by='score'
)
selections.append(sel)

# D5a: Farmyard positive
sel = find_reviews(
    has_farmyard & (df['score'] >= 85),
    'D5a: Farmyard as terroir (positive)', n=2, sort_by='score'
)
selections.append(sel)

# D5b: Farmyard negative
sel = find_reviews(
    has_farmyard & (df['score'] <= 78),
    'D5b: Farmyard as unclean (negative)', n=2, sort_by='score'
)
selections.append(sel)

# D6a: Oak as maturity
sel = find_reviews(
    (df['oak_review_text_per1k'] > oak_75) & (df['score'] >= 88),
    'D6a: Oak as maturity', n=2, sort_by='score'
)
selections.append(sel)

# D6b: Oak as excess
sel = find_reviews(
    has_overoaked,
    'D6b: Oak as excess (over-oaked)', n=2
)
selections.append(sel)

# D7: High complexity + low praise
sel = find_reviews(
    (df['structure_review_text_per1k'] > structure_p75) & (df['eval_review_text_per1k'] < eval_median),
    'D7: High complexity + low explicit praise', n=2, sort_by='structure_review_text_per1k'
)
selections.append(sel)

# D8: High praise + low complexity
sel = find_reviews(
    (df['eval_review_text_per1k'] > df['eval_review_text_per1k'].quantile(0.75)) &
    (df['structure_review_text_per1k'] < eval_median),
    'D8: High praise + low complexity', n=2, sort_by='eval_review_text_per1k'
)
selections.append(sel)

# D9: Nostalgia language
sel = find_reviews(
    has_nostalgia,
    'D9: Nostalgia language', n=2
)
selections.append(sel)

# Combine
candidates = pd.concat([s for s in selections if len(s) > 0], ignore_index=True)
candidates = candidates.drop_duplicates(subset='dedupe_hash')  # avoid same review appearing twice
print(); print(f"Total unique candidates selected: {len(candidates)}")

# Print summary table
print(); print("=== Close-Reading Candidates ===")
for i, row in candidates.iterrows():
    print(); print(f"[{row['selection_criterion']}]")
    print(f"  {row['whisky_name_raw']} ({row['distillery']}) — Score: {row['score']}")
    print(f"  Flaw: {row['flaw_review_text_per1k']:.1f}, Fruit: {row['fruit_review_text_per1k']:.1f}, Peat: {row['peat_review_text_per1k']:.1f}")
    # Print first 300 chars of review text
    text = str(row['review_text_original'])[:300]
    print(f"  Text: {text}...")

# Save
candidates.to_csv('data/w2_close_reading_candidates.csv', index=False)
print(); print(f"Saved {len(candidates)} candidates: data/w2_close_reading_candidates.csv")


Total unique candidates selected: 26

=== Close-Reading Candidates ===

[D1: High score + high flaw]
  Ardbeg 11 yo 1991/2002 (62.2%, Cadenhead's, Bond Reserve, bourbon hogshead, 306 bottles) (Ardbeg) — Score: 90
  Flaw: 43.0, Fruit: 5.4, Peat: 16.1
  Text: We were quite fond of this other series from Cadenhead. We've tried this before, utterly loved it, but never wrote any tasting note. Now is the time, twenty years later... Colour: straw. Nose: a strong presence of varnish, glue, cider vinegar, oysters, acetone, and that's about it at 62%, which is n...

[D1: High score + high flaw]
  Glenrothes 23 yo 1998/2022 'The 26 #1' (52.2%, Maltbarn, sherry cask, 47 bottles) (Glenrothes) — Score: 90
  Flaw: 37.7, Fruit: 0.0, Peat: 0.0
  Text: From Maltbarn's small-parcel line. Small outturns, excellent whiskies, as we've already noticed. And this, is sherry! Colour: deep gold. Nose: holy Molly, this is something! Litres of nail polish remover, a lot of balsamico, much more compost this time, 

### D3. Print Full Review Texts for Qualitative Reading

In [14]:
print("=" * 80)
print("FULL REVIEW TEXTS FOR CLOSE READING")
print("=" * 80)

for i, row in candidates.iterrows():
    print(); print(f"{'─'*80}")
    print(f"[{row['selection_criterion']}]")
    print(f"Whisky: {row['whisky_name_raw']}")
    print(f"Distillery: {row['distillery']} | Score: {row['score']} | Year: {row['review_year']}")
    print(f"{'─'*80}")
    print(row['review_text_original'])
    print()

FULL REVIEW TEXTS FOR CLOSE READING

────────────────────────────────────────────────────────────────────────────────
[D1: High score + high flaw]
Whisky: Ardbeg 11 yo 1991/2002 (62.2%, Cadenhead's, Bond Reserve, bourbon hogshead, 306 bottles)
Distillery: Ardbeg | Score: 90 | Year: 2023
────────────────────────────────────────────────────────────────────────────────
We were quite fond of this other series from Cadenhead. We've tried this before, utterly loved it, but never wrote any tasting note. Now is the time, twenty years later... Colour: straw. Nose: a strong presence of varnish, glue, cider vinegar, oysters, acetone, and that's about it at 62%, which is normal. With water: sharp lime, nail polish, more varnish, mop, old tweed, salty porridge, sourdough... Truly in the midst of nature. Mouth (neat): is this even legal? Acetone, glue, and kirsch fresh from the still. In short, wonderful promises... With water: roots, smoke and salt, candied turnips, sauerkraut, green pepper, samphi

## Summary: Results → Claims

### Claim 1: Expert valuation is not generic sentiment
- **Evidence:** M2 (Full Dictionary) vs M1 (VADER) R² comparison
- Dictionary Adj-R² > VADER Adj-R² → domain vocabulary captures evaluative content that generic sentiment misses
- M3 (without explicit eval) retains substantial R² → sensory/structural terms carry evaluative signal

### Claim 2: High value is associated with structure, not just pleasure
- **Evidence:** Coefficient signs on `structure` and `texture` in full model
- High-score group shows elevated structure/eval, low-score group shows elevated flaw

### Claim 3: Defects are culturally organized
- **Evidence:** Close-reading candidates (Task D) provide qualitative material
- Context-dependent terms (sulphur, farmyard, smoke, oak) cross the valued/devalued boundary

### Claim 4: Scores are textually legitimated
- **Evidence:** Section-level R² comparison (Task A3)
- NMF (sensory-only) retains signal comparable to Comments → evaluation embedded throughout